In [0]:
%sql

drop table if exists real_estate.gold.dim_zona;

create table real_estate.gold.dim_zona (
    zona_id bigint generated always as identity (start with 1 increment by 1),
    partido string not null,
    region string not null,
    ciudad string,
    provincia string default 'Buenos Aires',
    pais string default 'Argentina',
    _create_at timestamp default current_timestamp()
)
using delta
tblproperties(
    'delta.feature.allowColumnDefaults' = 'supported'
)
comment 'Dimensión de zonas - Star Schema';

insert into real_estate.gold.dim_zona (partido, region, ciudad, provincia, pais)
select distinct
    sp.partido,
    sp.region,
    case
        when sp.region = 'Capital federal' then 'CABA'
        else 'GBA'
    end as ciudad,
    'Buenos Aires' as provincia,
    'Argentina' as pais
from real_estate.silver.propiedades sp
where sp.partido is not null
order by sp.partido;

In [0]:
%sql
drop table if exists real_estate.gold.dim_tipo_operacion;

create table real_estate.gold.dim_tipo_operacion (
    tipo_operacion_id bigint generated always as identity (start with 1 increment by 1), 
    tipo_operacion string not null, 
    moneda string not null, 
    categoria string, 
    descripcion string,
    _create_at timestamp default current_timestamp()
)
using delta
tblproperties (
    'delta.feature.allowColumnDefaults' = 'supported'
)
comment 'Dimensión de tipo de operación + moneda - Star Schema';

insert into table real_estate.gold.dim_tipo_operacion (tipo_operacion, moneda, categoria, descripcion)
select distinct
    tipo_de_operacion,
    moneda,
    case 
        when tipo_de_operacion = 'alquiler' then 'residencial'
        when tipo_de_operacion = 'venta' then 'residencial'
        when tipo_de_operacion = 'alquiler_temporario' then 'temporal'
        else 'otro'
    end as categoria,
    case 
        when tipo_de_operacion = 'arquiler' then 'Alquiler residencial'
        when tipo_de_operacion = 'venta' then 'Venta de propiedad'
        when tipo_de_operacion = 'alquiler_temporario' then 'Alquiler temporal'
        else 'otros'
    end as descripcion
    from real_estate.silver.propiedades
    where tipo_de_operacion is not null and moneda is not null
    order by tipo_de_operacion, moneda

In [0]:
%sql

drop table if exists real_estate.gold.dim_tiempo;

create table real_estate.gold.dim_tiempo (
    fecha_id bigint,
    fecha date not null,
    anio int,
    mes int,
    trimestre int,
    dia_semana string,
    es_fin_de_semana boolean,
    _create_at timestamp default current_timestamp()
)
using delta 
tblproperties (
    'delta.feature.allowColumnDefaults' = 'supported'
)
comment 'Dimensión de tiempo - Star Schema';

insert into real_estate.gold.dim_tiempo (fecha_id, fecha, anio, mes, trimestre, dia_semana, es_fin_de_semana)
select distinct
    cast(date_format(fecha_publicacion, 'yyyyMMdd') as bigint) as fecha_id,
    fecha_publicacion as fecha,
    year(fecha_publicacion) as anio,
    month(fecha_publicacion) as mes,
    quarter(fecha_publicacion) as trimestre,
    case dayofweek(fecha_publicacion)
        when 1 then 'Domingo'
        when 2 then 'Lunes'
        when 3 then 'Martes'
        when 4 then 'Miercoles'
        when 5 then 'Jueves'
        when 6 then 'Viernes'
        when 7 then 'Sabado'
    end as dia_semana,
    dayofweek(fecha_publicacion) in (1, 7) as es_fin_de_semana
from real_estate.silver.propiedades
where fecha_publicacion is not null

In [0]:
%sql
drop table if exists real_estate.gold.dim_caracteristicas;

create table real_estate.gold.dim_caracteristicas (
    caracteristicas_id bigint generated always as identity (start with 1 increment by 1),
    estado string not null,
    cochera boolean,
    _create_at timestamp default current_timestamp()
)
using delta
tblproperties (
    'delta.feature.allowColumnDefaults' = 'supported'
)
comment 'Dimensión de características de la propiedad (estado + cochera) - Star Schema';

insert into real_estate.gold.dim_caracteristicas (estado, cochera)
select distinct
    coalesce(estado, 'sin espesificar') as estado,
    coalesce(cochera, false) as cochera
from real_estate.silver.propiedades
order by estado, cochera

In [0]:
%sql
drop table if exists real_estate.gold.dim_orientacion;

create table real_estate.gold.dim_orientacion (
    orientacion_id bigint generated always as identity (start with 1 increment by 1),
    orientacion string not null,
    tipo_orientacion string,
    _create_at timestamp default current_timestamp()
)
using delta
tblproperties (
    'delta.feature.allowColumnDefaults' = 'supported'
    
)
comment 'Dimensión de orientación - Star Schema';

insert into real_estate.gold.dim_orientacion (orientacion, tipo_orientacion)
select distinct
    coalesce(orientacion, 'sin especificar') as orientacion,
    case 
        when lower(orientacion) like '%norte%' then 'norte'
        when lower(orientacion) like '%sur%' then 'sur'
        when lower(orientacion) like '%este%' then 'este'
        when lower(orientacion) like '%oeste%' then 'oeste'
        else 'sin especificar'
    end as tipo_orientacion
from real_estate.silver.propiedades
order by orientacion;

In [0]:
%sql

drop